# Part 5: Arbitrary MCP servers (Microsoft Learn, etc.)

This notebook shows how to wire up **arbitrary MCP servers** as knowledge sources.
We use **Microsoft Learn** (`https://learn.microsoft.com/api/mcp`) as the primary
example.

### Prerequisites

| Item | Details |
|------|---------|
| `.env` file | In the `notebooks/` directory (same folder as this notebook) — `BIGBUGS_ENDPOINT`, `BIGBUGS_API_KEY`, `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_KEY`. |

> ⚠️ **The `.env` is currently in testing state — ask Matt for setup credentials.**
> Never check secrets into this repo.

### Current status on BigBugs

| Step | Status |
|------|--------|
| Direct MCP probe (initialize/tools/list/tools/call) | ✅ Works for Learn |
| `mcpServer` KS create | ✅ Works |
| `mcpServer` KB create | ✅ Works (needs `retrievalReasoningEffort = low` + chat model like `gpt-5.4-mini`) |
| KB retrieve with `knowledgeSourceParams.kind = "mcpServer"` | ❌ **500** — model-copy deserialization bug |
| KB retrieve without ksParams | ⚠️ Returns 200 but only **stub** content (`"Stub MCP server result"`) |

## 1 — Setup

In [ ]:
import json, os
from pathlib import Path
from datetime import datetime, timezone

import requests

ENV_PATH = Path(".").resolve() / ".env"

def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        values[k.strip()] = v.strip()
    return values

env = load_env(ENV_PATH)
ENDPOINT = env["BIGBUGS_ENDPOINT"].rstrip("/")
API_KEY  = env["BIGBUGS_API_KEY"]
API_VERSION = "2026-05-01-preview"

LEARN_MCP_ENDPOINT = env.get("MSLEARN_ENDPOINT", "https://learn.microsoft.com/api/mcp").rstrip("/")
AZURE_OPENAI_ENDPOINT = env.get("AZURE_OPENAI_ENDPOINT", "").rstrip("/") + "/"
AZURE_OPENAI_KEY = env.get("AZURE_OPENAI_KEY", "")

session = requests.Session()
session.headers.update({"api-key": API_KEY, "Accept": "application/json"})

def url(path: str) -> str:
    sep = "&" if "?" in path else "?"
    return f"{ENDPOINT}{path}{sep}api-version={API_VERSION}"

def pp(r: requests.Response):
    print(f"{r.status_code} {r.reason}")
    try:
        print(json.dumps(r.json(), indent=2))
    except Exception:
        print(r.text[:500])

stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
KS_NAME = f"lab532-learn-mcp-{stamp}"
KB_NAME = f"lab532-learn-mcp-kb-{stamp}"

print(f"Search Endpoint : {ENDPOINT}")
print(f"Learn MCP       : {LEARN_MCP_ENDPOINT}")
print(f"KS              : {KS_NAME}")
print(f"KB              : {KB_NAME}")

## 2 — Probe Microsoft Learn MCP directly

Validate the upstream server responds to the MCP protocol.

In [ ]:
learn_session = requests.Session()
learn_session.headers.update({"Accept": "application/json, text/event-stream"})

# Initialize
r = learn_session.post(LEARN_MCP_ENDPOINT, json={
    "jsonrpc": "2.0", "id": 1,
    "method": "initialize",
    "params": {
        "protocolVersion": "2024-11-05",
        "capabilities": {},
        "clientInfo": {"name": "lab532-notebook", "version": "1.0"},
    },
}, headers={"Content-Type": "application/json"})
print("=== initialize ===")
pp(r)

mcp_session_id = r.headers.get("mcp-session-id", "")
extra = {"mcp-session-id": mcp_session_id} if mcp_session_id else {}

In [ ]:
# Initialized notification
learn_session.post(LEARN_MCP_ENDPOINT, json={
    "jsonrpc": "2.0", "method": "notifications/initialized", "params": {},
}, headers={"Content-Type": "application/json", **extra})

# tools/list
r = learn_session.post(LEARN_MCP_ENDPOINT, json={
    "jsonrpc": "2.0", "id": 2,
    "method": "tools/list", "params": {},
}, headers={"Content-Type": "application/json", **extra})
print("=== tools/list ===")
pp(r)

# Find the docs search tool
tool_name = "microsoft_docs_search"
try:
    tools = r.json()["result"]["tools"]
    for t in tools:
        if "docs_search" in t.get("name", "").lower():
            tool_name = t["name"]
            break
    print(f"\nResolved tool: {tool_name}")
except Exception:
    print(f"\nUsing default tool name: {tool_name}")

In [ ]:
# tools/call
r = learn_session.post(LEARN_MCP_ENDPOINT, json={
    "jsonrpc": "2.0", "id": 3,
    "method": "tools/call",
    "params": {
        "name": tool_name,
        "arguments": {"query": "Azure AI Search knowledge base knowledge source preview"},
    },
}, headers={"Content-Type": "application/json", **extra}, timeout=60)
print("=== tools/call ===")
pp(r)

## 3 — Create an `mcpServer` knowledge source pointing to Learn

In [ ]:
ks_body = {
    "name": KS_NAME,
    "kind": "mcpServer",
    "description": "Lab 532 Microsoft Learn MCP",
    "mcpServerParameters": {
        "serverURL": LEARN_MCP_ENDPOINT,
        "tools": [
            {"name": tool_name, "outputParsing": {"kind": "auto"}}
        ],
    },
}

r = session.put(url(f"/knowledgesources('{KS_NAME}')"), json=ks_body,
                headers={"Prefer": "return=representation"})
print("=== Create mcpServer KS ===")
pp(r)

In [ ]:
# Check status
r = session.get(url(f"/knowledgesources('{KS_NAME}')/status"))
print("=== KS Status ===")
pp(r)

## 4 — Create a KB with a chat model

`mcpServer` KBs require:
- `retrievalReasoningEffort: {kind: "low"}` (not `minimal`)
- A valid Azure OpenAI chat deployment (`gpt-5.4-mini` works)

In [ ]:
kb_body = {
    "name": KB_NAME,
    "description": "Lab 532 KB — Learn MCP",
    "outputMode": "extractiveData",
    "retrievalReasoningEffort": {"kind": "low"},
    "retrievalInstructions": "Use the Microsoft Learn MCP tool for up-to-date documentation.",
    "knowledgeSources": [{"name": KS_NAME}],
    "models": [
        {
            "kind": "azureOpenAI",
            "azureOpenAIParameters": {
                "resourceUri": AZURE_OPENAI_ENDPOINT,
                "deploymentId": "gpt-5.4-mini",
                "modelName": "gpt-5.4-mini",
                "apiKey": AZURE_OPENAI_KEY,
            },
        }
    ],
}

r = session.put(url(f"/knowledgebases('{KB_NAME}')"), json=kb_body,
                headers={"Prefer": "return=representation"})
print("=== Create KB ===")
pp(r)

## 5 — Retrieve (expected to fail on current stamp)

### Attempt 1: with `knowledgeSourceParams`
This sends `kind: "mcpServer"` in the retrieve params — expected to return **500**
due to a server-side model-copy deserialization bug.

In [ ]:
retrieve_body = {
    "includeActivity": True,
    "messages": [
        {"role": "user",
         "content": [{"type": "text", "text": "What is new in Azure AI Search knowledge bases?"}]}
    ],
    "knowledgeSourceParams": [
        {
            "kind": "mcpServer",
            "knowledgeSourceName": KS_NAME,
            "includeReferenceSourceData": True,
            "includeReferences": True,
            "alwaysQuerySource": True,
        }
    ],
    "maxRuntimeInSeconds": 120,
}

r = session.post(url(f"/knowledgebases('{KB_NAME}')/retrieve"), json=retrieve_body, timeout=180)
print("=== Retrieve with ksParams (expect 500) ===")
pp(r)

if r.status_code == 500:
    print("\n❌ Expected failure: model-copy deserialization bug for mcpServer kind.")
    print("   Tracked as a known product gap on the BigBugs stamp.")

### Attempt 2: without `knowledgeSourceParams`
This omits the per-source params — may return 200 but with only **stub** content.

In [ ]:
retrieve_no_params = {
    "includeActivity": True,
    "messages": [
        {"role": "user",
         "content": [{"type": "text", "text": "What is new in Azure AI Search knowledge bases?"}]}
    ],
    "maxRuntimeInSeconds": 120,
}

r = session.post(url(f"/knowledgebases('{KB_NAME}')/retrieve"), json=retrieve_no_params, timeout=180)
print("=== Retrieve without ksParams ===")
pp(r)

if r.status_code == 200:
    body = r.json()
    for item in body.get("response", []):
        for c in item.get("content", []):
            text = c.get("text", "")
            if "Stub" in text:
                print("\n⚠️ Stub content returned — real MCP retrieval path not wired yet.")
            else:
                print(f"\nResponse text: {text[:200]}")

## 6 — Cleanup

In [ ]:
for label, path in [
    ("KB", f"/knowledgebases('{KB_NAME}')"),
    ("KS", f"/knowledgesources('{KS_NAME}')"),
]:
    r = session.delete(url(path))
    print(f"Delete {label}: {r.status_code} {r.reason}")

## Summary & known blockers

| What | Works? | Notes |
|------|--------|-------|
| Direct upstream MCP probe | ✅ | Learn + MAI Grounding both respond correctly |
| `mcpServer` KS CRUD | ✅ | Create, read, status, delete all work |
| `mcpServer` KB create | ✅ | Needs `low` reasoning + valid chat model |
| KB retrieve **with** `mcpServer` ksParams | ❌ | 500 — server-side deserialization bug |
| KB retrieve **without** ksParams | ⚠️ | 200 but returns stub content, not real MCP output |
| `retrievalReasoningEffort = minimal` | ❌ | Not supported for `mcpServer` KBs |
| `gpt-4o-mini` model | ❌ | Not available on the configured Foundry resource |
| `gpt-5.4-mini` model | ✅ | Works for KB create |

Once the product team fixes the retrieve-path model-copy bug, the full end-to-end
flow (KS → KB → retrieve → real MCP grounding) should work.

## Wrap-up

You now have the five-part notebook set covering:

1. **Search Index** — fully working
2. **Fabric IQ** — fully working (needs Fabric token)
3. **Work IQ** — works but intermittent
4. **Web / MAI Grounding** — direct MCP works, KB retrieve blocked
5. **Arbitrary MCP** — KS+KB setup works, retrieve blocked

Previous MVP Summit notebooks are preserved under `notebooks/mvp-summit-notebooks/`.